# Recurrent parallelism sweep — Braille SRNN

Reproduces the experiment of Section 5.5.2 of the dissertation on the two
published Braille CUBA-LIF graphs, each compared against an event-driven
active-list implementation of the same graph.

**Why this one is indexed by `p` and not by an element count.** In the
feed-forward network one dense layer carries 98.6% of the static work, so a
single resolved count describes the design. Here the dominant layer is the
recurrent transform and it carries 64%, so parallelising it alone would
saturate at 2.8x by Amdahl's law. The request is model-wide and every layer
resolves its own count from its own work domain.

**Cost.** About four hours for the twenty time-driven points plus the two
event-driven references. Unlike the feed-forward sweep, the full range
`p` in [0, 1] is reachable, because the dominant layer holds 1 444 operations
rather than 100 352.

In [ ]:
import json, subprocess, sys
from pathlib import Path

AQUI = Path.cwd()
RAIZ = AQUI.parent
RUNS = RAIZ / "sim" / "runs"
PY = sys.executable          # o mesmo interpretador do kernel roda os scripts

def executar(args, **kw):
    """Roda um comando na raiz do repositorio e devolve o codigo de saida."""
    print("$", " ".join(str(a) for a in args))
    return subprocess.run(args, cwd=RAIZ, **kw).returncode

def ultimo_resumo(componente):
    """summary.json da run mais recente que produziu resultado, ou None."""
    achados = sorted((RUNS / componente).glob("*/reports/summary.json")) \
        if (RUNS / componente).is_dir() else []
    return json.loads(achados[-1].read_text()) if achados else None

print("repositorio:", RAIZ)
print("python     :", PY)
for f in ("vitis-run", "vivado"):
    r = subprocess.run(["which", f], capture_output=True, text=True)
    print(f"{f:11s}:", r.stdout.strip() or "NAO ENCONTRADO no PATH")

# --- configuracao do sweep, num lugar so ---------------------------------
GRAFOS = ["zero", "subtract"]
PONTOS = ["00000", "00050", "00100", "00200", "00400",
          "00800", "01600", "03200", "06400", "10000"]
FORCAR = False        # True regenera componentes que ja existem
SO_FALTANTES = True   # False refaz pontos que ja tem resultado
TIMEOUT_MIN = 300     # por componente


## The sweep

Ten points per graph, doubling from 0.005, plus the serial request. Each step
roughly halves the summed reuse count and no two points resolve to the same
architecture — `p = 0.7`, for instance, is excluded because it produces the
same plan as `p = 0.5`.

`p = 0` is kept here, unlike in the feed-forward sweep: it is the only fully
serial point and does not duplicate its neighbour, which already resolves to
seven elements on the recurrent layer.

The same call also emits the two event-driven references, which use the
active-list strategy with a noise threshold of 1e-6.

In [ ]:
cmd = [PY, "SRNN_test/gerar_componentes_percent_parallelism.py", "--all"]
if FORCAR:
    cmd.append("--force")
executar(cmd)

## The resolved plan

The vector below is the point of this experiment. A single number cannot stand
for the design: at `p = 1` the six layers resolve to 456, 38, 38, 1 444, 266
and 7 elements respectively, and the recurrent layer exceeds its own output
count long before that — which is what the reduction lanes exist to absorb.

In [ ]:
for g in GRAFOS:
    print(f"\n--- Braille {g} ---")
    for tag in PONTOS:
        m = AQUI / f"hls_time_driven_{g}_p{tag}" / "parallelism_manifest.json"
        if not m.is_file():
            continue
        camadas = json.loads(m.read_text())["layers"]
        p = camadas[0]["requested_parallelism"]
        u = [c["processing_elements"] for c in camadas]
        print(f"  p={p:<7g} U por camada = {u}")

## Running the pipeline

The two backends need different settings, so each has its own environment
file: the event-driven reference reports a data-dependent latency and has no
usable synthesis figure, so only it runs co-simulation.

The loop stops a graph at its first failure. Points above a failed one request
strictly more hardware, so continuing would spend hours collecting the same
outcome.

In [ ]:
import shutil


def rodar(proj, env):
    comp = Path(proj).name
    if SO_FALTANTES and ultimo_resumo(comp):
        print(f"{comp}: ja medido, pulando"); return 0
    rc = executar(["timeout", f"{TIMEOUT_MIN}m", PY, "-m", "sim", "run",
                   "--project", proj, "--to", "power", "--environment", env])
    for projeto in (RUNS / comp).glob("*/project"):
        shutil.rmtree(projeto, ignore_errors=True)
    print(f"{comp}: exit={rc}")
    return rc

for g in GRAFOS:
    for tag in PONTOS:
        if rodar(f"SRNN_test/hls_time_driven_{g}_p{tag}",
                 "SRNN_test/environment_td.yaml") != 0:
            print(f"parando o grafo {g}"); break
    rodar(f"SRNN_test/hls_event_driven_{g}_active_list",
          "SRNN_test/environment_ed.yaml")

## Results

Each graph is read against its own event-driven reference. The crossing arrives
early — the zero-reset graph passes it at `p = 0.01` — and beyond it the
time-driven component wins on latency and energy at once, and on logic as well
at two of the points.

In [ ]:
def metricas(comp):
    s = ultimo_resumo(comp)
    if not s:
        return None
    passos = (s.get("workload") or {}).get("total_logical_steps")
    cos = (s.get("cosim") or {}).get("total_execution_cycles")
    lat = cos / passos if cos and passos else float(s["hls"]["latency"]["worst_case"])
    r, pw = s["vivado_utilization"]["resources"], s["power"]
    t = pw["total_on_chip_power_w"]
    return lat, r["lut"]["used"], r["dsp"]["used"], t * lat / 150e6 * 1e6

for g in GRAFOS:
    ed = metricas(f"hls_event_driven_{g}_active_list")
    print(f"\n--- Braille {g} "
          f"(ED: {ed[0]:,.0f} ciclos, {ed[1]:,.0f} LUT, {ed[3]:.2f} uJ)" if ed
          else f"\n--- Braille {g} (sem referencia ED)")
    print(f"  {'p':>7s} {'ciclos':>8s} {'LUT':>9s} {'DSP':>7s} {'uJ/passo':>9s} {'vs ED':>7s}")
    for tag in PONTOS:
        m = metricas(f"hls_time_driven_{g}_p{tag}")
        if not m:
            continue
        p = int(tag) / 10000
        rel = f"{ed[0]/m[0]:6.2f}x" if ed else "     --"
        print(f"  {p:7g} {m[0]:8,.0f} {m[1]:9,.0f} {m[2]:7,.0f} {m[3]:9.2f} {rel:>7s}")

## Report and figures

The report writes one markdown table per graph and four figures — latency with
energy, and the four resources — in the same style as the feed-forward report,
so the two experiments can be read side by side.

In [ ]:
executar([PY, "SRNN_test/gerar_relatorio_percent_parallelism.py"])

from IPython.display import Image, Markdown, display
for g in GRAFOS:
    for nome in ("tempo_energia", "recursos"):
        alvo = AQUI / f"relatorio_percent_parallelism_{g}_{nome}.png"
        if alvo.is_file():
            display(Image(filename=str(alvo)))
display(Markdown((AQUI / "relatorio_percent_parallelism.md").read_text()[:1500] + "\n\n..."))